## Setup, Memory Monitoring

In [1]:
import gc
import os
import numpy as np
import pandas as pd
import psutil

pd.set_option('display.max_columns', None)

def show_memory(label=""):
    """Print current process RSS memory -- lets us verify each optimization actually helps."""
    mem_gb = psutil.Process(os.getpid()).memory_info().rss / 1e9
    print(f"[{label}] Process memory: {mem_gb:.2f} GB")

show_memory("notebook start")

[notebook start] Process memory: 0.17 GB


## Load with Dtypes Set Upfront

In [2]:
from google.colab import drive
drive.mount('/content/drive')

DATA_PATH = "/content/drive/MyDrive/Supply_Chain_Demand_Forecasting/data/processed/cleaned_sales_data.csv"

dtype_map = {
    'store_id': 'category', 'item_id': 'category', 'dept_id': 'category',
    'cat_id': 'category', 'state_id': 'category',
    'event_name_1': 'category', 'event_type_1': 'category',
    'event_name_2': 'category', 'event_type_2': 'category',
    'day_name': 'category',
    'wday': 'int8', 'month': 'int8', 'snap': 'int8', 'is_zero_sale': 'int8',
}

df = pd.read_csv(DATA_PATH, parse_dates=['date'], dtype=dtype_map)
df = df.sort_values(['store_id', 'item_id', 'date']).reset_index(drop=True)

show_memory("after load")
print(df.shape)
df.dtypes

Mounted at /content/drive
[after load] Process memory: 1.65 GB
(9586222, 22)


,0
id,object
item_id,category
dept_id,category
cat_id,category
store_id,category
state_id,category
d,object
units_sold,int64
date,datetime64[ns]
wm_yr_wk,int64


## Downcast Remaining Numeric Columns

In [3]:
def downcast_numeric(frame, int_cols=None, float_cols=None):
    """Downcast integer/float columns to the smallest dtype that safely holds their values."""
    if int_cols:
        for c in int_cols:
            frame[c] = pd.to_numeric(frame[c], downcast='integer')
    if float_cols:
        for c in float_cols:
            frame[c] = pd.to_numeric(frame[c], downcast='float')
    return frame

df = downcast_numeric(df, int_cols=['units_sold', 'year', 'wm_yr_wk'], float_cols=['sell_price'])

show_memory("after downcast")
df.info(memory_usage='deep')

[after downcast] Process memory: 1.39 GB
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 9586222 entries, 0 to 9586221
Data columns (total 22 columns):
 #   Column        Dtype         
---  ------        -----         
 0   id            object        
 1   item_id       category      
 2   dept_id       category      
 3   cat_id        category      
 4   store_id      category      
 5   state_id      category      
 6   d             object        
 7   units_sold    int16         
 8   date          datetime64[ns]
 9   wm_yr_wk      int16         
 10  weekday       object        
 11  wday          int8          
 12  month         int8          
 13  year          int16         
 14  event_name_1  category      
 15  event_type_1  category      
 16  event_name_2  category      
 17  event_type_2  category      
 18  sell_price    float32       
 19  snap          int8          
 20  is_zero_sale  int8          
 21  day_name      category      
dtypes: category(10), datetime

## Lag, Rolling, and EMA Features (Cached Groupby + transform)

In [4]:
LAG_DAYS = [1, 7, 14, 28]
ROLLING_WINDOWS = [7, 14, 28]
EMA_SPANS = [7, 14]

def add_lag_rolling_ema_features(store_df):
    """Lag/rolling/EMA features for one store, using a single cached groupby object
    and .transform() -- avoids the manual multi-index rebuild that drove up memory before."""
    g = store_df.groupby('item_id', observed=True, sort=False)['units_sold']

    for lag in LAG_DAYS:
        store_df[f'lag_{lag}'] = g.shift(lag).astype('float32')

    # Shift once, reuse the shifted column's groupby for every rolling/EMA stat below
    store_df['_shift1'] = g.shift(1)
    g_shift = store_df.groupby('item_id', observed=True, sort=False)['_shift1']

    for w in ROLLING_WINDOWS:
        store_df[f'rolling_mean_{w}'] = g_shift.transform(lambda s: s.rolling(w).mean()).astype('float32')
        store_df[f'rolling_std_{w}'] = g_shift.transform(lambda s: s.rolling(w).std()).astype('float32')
        store_df[f'rolling_median_{w}'] = g_shift.transform(lambda s: s.rolling(w).median()).astype('float32')

    for span in EMA_SPANS:
        store_df[f'ema_{span}'] = g_shift.transform(lambda s: s.ewm(span=span, adjust=False).mean()).astype('float32')

    store_df.drop(columns=['_shift1'], inplace=True)
    return store_df

## Calendar and Cyclic Features (Vectorized, No Grouping Needed)

In [5]:
def add_calendar_features(store_df):
    """Pure vectorized calendar + cyclic encoding -- cheap, no groupby required."""
    dow = store_df['date'].dt.dayofweek.astype('int8')
    store_df['day_of_week_num'] = dow
    store_df['day_of_month'] = store_df['date'].dt.day.astype('int8')
    store_df['month_num'] = store_df['date'].dt.month.astype('int8')
    store_df['week_of_year'] = store_df['date'].dt.isocalendar().week.astype('int8')
    store_df['is_weekend'] = (dow >= 5).astype('int8')

    store_df['dow_sin'] = np.sin(2 * np.pi * dow / 7).astype('float32')
    store_df['dow_cos'] = np.cos(2 * np.pi * dow / 7).astype('float32')
    store_df['month_sin'] = np.sin(2 * np.pi * store_df['month_num'] / 12).astype('float32')
    store_df['month_cos'] = np.cos(2 * np.pi * store_df['month_num'] / 12).astype('float32')
    return store_df

## SNAP, Event, and Price Features (Efficient snap_lead)

In [6]:
# ============================================================
# ROOT-CAUSE FIX for the snap_lead NaN bug (was: Section 5, original version)
# ============================================================
# ORIGINAL BUG: snap_lead looked ahead only within store_df['snap'], which is
# truncated to the sales file's date range (d_1..d_1941). calendar.csv actually
# publishes real SNAP flags through d_1969 -- that extra, already-known
# government schedule data was never merged in, so rows near the tail of
# d_1941 had nothing left to look ahead to and became NaN.
#
# FIX: build a (state_id, date) -> snap_lead lookup ONCE from the FULL,
# untruncated calendar.csv, then assign it via merge instead of a per-item
# lookahead over a truncated column. This:
#   - uses only real, already-published SNAP data (no leakage)
#   - is computed once per state instead of redundantly per (store, item),
#     since SNAP is identical across every item in a state
#   - right-censors the true residual tail (the very end of calendar.csv
#     itself) at the max observed lead, instead of fabricating a fill value

PROJECT_DIR = "/content/drive/MyDrive/Supply_Chain_Demand_Forecasting"
RAW_DIR = f"{PROJECT_DIR}/data/raw"
CALENDAR_PATH = f"{RAW_DIR}/calendar.csv"

def days_until_next_snap_full(snap_series):
    """Days until the next SNAP day, computed over a FULL, untruncated calendar
    range. Right-censors any true residual tail (beyond calendar.csv's own last
    date) at the max observed lead value, instead of leaving NaN or fabricating 0."""
    arr = np.asarray(snap_series)
    n = len(arr)
    true_idx = np.flatnonzero(arr == 1)
    result = np.full(n, np.nan, dtype='float32')
    if true_idx.size == 0:
        return result

    positions = np.arange(n)
    pos = np.searchsorted(true_idx, positions)
    valid = pos < true_idx.size
    result[valid] = (true_idx[pos[valid]] - positions[valid]).astype('float32')

    if (~valid).any():
        censor_value = np.nanmax(result) if valid.any() else 30.0
        result[~valid] = censor_value

    return result


# Build the lookup ONCE, before the per-store loop -- state-level, not per item,
# and over the FULL calendar (not the truncated sales-file range).
_calendar_full = pd.read_csv(CALENDAR_PATH, parse_dates=['date'])
_calendar_full = _calendar_full.sort_values('date').reset_index(drop=True)

_state_snap_cols = {'CA': 'snap_CA', 'TX': 'snap_TX', 'WI': 'snap_WI'}
_lookup_frames = []
for state_id, snap_col in _state_snap_cols.items():
    lead_values = days_until_next_snap_full(_calendar_full[snap_col].values)
    _lookup_frames.append(pd.DataFrame({
        'state_id': state_id,
        'date': _calendar_full['date'],
        'snap_lead': lead_values.astype('float32')
    }))

SNAP_LEAD_LOOKUP = pd.concat(_lookup_frames, ignore_index=True)
del _calendar_full, _lookup_frames
gc.collect()

assert SNAP_LEAD_LOOKUP['snap_lead'].isnull().sum() == 0, \
    "snap_lead lookup still has NaN -- check calendar.csv SNAP columns."
print("snap_lead lookup built over the FULL calendar range. Rows:", SNAP_LEAD_LOOKUP.shape[0])


def add_event_snap_price_features(store_df):
    """Event, SNAP-lead (via full-calendar lookup), and price features.
    sell_price is used same-day intentionally -- price is known in advance,
    so this is not leakage."""
    store_df['has_event'] = (store_df['event_name_1'] != 'none').astype('int8')

    # snap_lead now assigned via merge against the FULL-range lookup, not a
    # per-item transform over a truncated column -- fixes the Fold 4 NaN bug
    # at its source.
    store_df = store_df.merge(SNAP_LEAD_LOOKUP, on=['state_id', 'date'], how='left')

    g_price = store_df.groupby('item_id', observed=True, sort=False)['sell_price']
    store_df['_shift_price'] = g_price.shift(1)

    g_shift_price = store_df.groupby('item_id', observed=True, sort=False)['_shift_price']
    store_df['price_rolling_mean_28'] = (
        g_shift_price.transform(lambda s: s.rolling(28, min_periods=1).mean()).astype('float32')
    )
    store_df['price_relative'] = (
        store_df['sell_price'] / store_df['price_rolling_mean_28']
    ).astype('float32')
    store_df['price_changed'] = (
        g_price.diff().fillna(0) != 0
    ).astype('int8')

    store_df.drop(columns=['_shift_price'], inplace=True)
    return store_df

snap_lead lookup built over the FULL calendar range. Rows: 5907


## Per-Store Processing Loop (Caps Peak Memory)

In [7]:
TEMP_DIR = '/content/feature_chunks/'
os.makedirs(TEMP_DIR, exist_ok=True)

stores = df['store_id'].cat.categories.tolist()
print("Processing stores:", stores)

for store in stores:
    show_memory(f"before {store}")

    store_df = df[df['store_id'] == store].copy()
    store_df = store_df.sort_values(['item_id', 'date']).reset_index(drop=True)

    store_df = add_lag_rolling_ema_features(store_df)
    store_df = add_calendar_features(store_df)
    store_df = add_event_snap_price_features(store_df)

    # Same policy as before: require full lookback history (lag_28 / rolling_mean_28) before keeping a row
    before_rows = len(store_df)
    store_df.dropna(subset=['lag_28', 'rolling_mean_28'], inplace=True)
    store_df.reset_index(drop=True, inplace=True)
    print(f"{store}: dropped {before_rows - len(store_df)} rows lacking full lookback "
          f"({(before_rows - len(store_df)) / before_rows * 100:.2f}%)")

    store_df.to_parquet(f"{TEMP_DIR}{store}_features.parquet", index=False)

    del store_df
    gc.collect()
    show_memory(f"after {store}, freed from memory")

print("\nAll store chunks processed and written to disk.")

Processing stores: ['CA_1', 'TX_1']
[before CA_1] Process memory: 1.39 GB
CA_1: dropped 85372 rows lacking full lookback (1.78%)
[after CA_1, freed from memory] Process memory: 1.56 GB
[before TX_1] Process memory: 1.56 GB
TX_1: dropped 85372 rows lacking full lookback (1.78%)
[after TX_1, freed from memory] Process memory: 1.54 GB

All store chunks processed and written to disk.


## Concatenate Chunks and Final Save

In [8]:
chunk_files = [TEMP_DIR + f for f in os.listdir(TEMP_DIR) if f.endswith('.parquet')]
print("Chunks found:", chunk_files)

df_features = pd.concat([pd.read_parquet(f) for f in chunk_files], ignore_index=True)
del chunk_files
gc.collect()

show_memory("after concat")
print("Final feature matrix shape:", df_features.shape)

Chunks found: ['/content/feature_chunks/CA_1_features.parquet', '/content/feature_chunks/TX_1_features.parquet']
[after concat] Process memory: 4.26 GB
Final feature matrix shape: (9415478, 51)


In [9]:
# Final safety-net downcast in case concatenation upcast anything
float_cols = df_features.select_dtypes(include='float64').columns.tolist()
int_cols = df_features.select_dtypes(include='int64').columns.tolist()
df_features = downcast_numeric(df_features, int_cols=int_cols, float_cols=float_cols)

show_memory("after final downcast")
df_features.info(memory_usage='deep')

[after final downcast] Process memory: 4.26 GB
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 9415478 entries, 0 to 9415477
Data columns (total 51 columns):
 #   Column                 Dtype         
---  ------                 -----         
 0   id                     object        
 1   item_id                category      
 2   dept_id                category      
 3   cat_id                 category      
 4   store_id               category      
 5   state_id               object        
 6   d                      object        
 7   units_sold             int16         
 8   date                   datetime64[ns]
 9   wm_yr_wk               int16         
 10  weekday                object        
 11  wday                   int8          
 12  month                  int8          
 13  year                   int16         
 14  event_name_1           category      
 15  event_type_1           category      
 16  event_name_2           category      
 17  event_type_2      

In [11]:
OUT_PATH = '/content/drive/MyDrive/Supply_Chain_Demand_Forecasting/data/processed/feature_engineered_data.csv'

df_features.to_csv(OUT_PATH, index=False)
show_memory("after save")
print("Saved feature_engineered_data.csv —", df_features.shape)

[after save] Process memory: 4.26 GB
Saved feature_engineered_data.csv — (9415478, 51)


##Key Takeaways
###Successfully engineered predictive time-series features including lag, rolling statistics, exponential moving averages (EMA), calendar, cyclic, SNAP, event, and price-based features.
###Optimized the feature engineering pipeline for Google Colab Free using memory-efficient data types, downcasting, per-store processing, and intermediate Parquet storage.
###Reduced peak memory consumption by processing stores independently instead of loading the entire feature matrix into memory.
###Generated a complete feature matrix suitable for machine learning models while preserving chronological order.